In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import json

In [2]:
file_path = '../../results/data_results/cifar10_all_layers/cifar10_results1.jsonl'

In [3]:
plot_epochs = [5, 10, 15]

results = []
epoch_data = {}
with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        epoch = item['epoch']
        mean_acc = item['mean_acc']
        mean_f1 = item['mean_f1']

        if epoch in plot_epochs:
            epoch_data[epoch] = {
                "mean_f1": mean_f1,
                "mean_acc": mean_acc,
                "distance": item['distance'],
            }

plot_epochs_found = [e for e in plot_epochs if e in epoch_data]
missing_epochs = [e for e in plot_epochs if e not in epoch_data]
if missing_epochs:
    print(f"Warning: missing epochs: {missing_epochs}")

plot_f1s = [epoch_data[e]['mean_f1'] for e in plot_epochs_found]
plot_accs = [epoch_data[e]['mean_acc'] for e in plot_epochs_found]


In [4]:
plot_epochs


[5, 10, 15]

In [5]:
plot_epochs_found


[5, 10, 15]

In [6]:
missing_epochs


[]

In [7]:
plot_f1s


[0.4988321363925934, 0.6463913917541504, 0.6720167398452759]

In [8]:
plot_accs


[0.5163, 0.6494, 0.678]

In [9]:
len(plot_epochs_found)


3

In [10]:
plot_epochs_found


[5, 10, 15]

In [11]:
missing_epochs


[]

In [12]:
def get_plot_dict(data):
    intra = data['intra']
    inter = data['inter']

    layers = ['input', 
              'layer1_block0', 'layer1_block1', 'layer1_block2',
              'layer1', 
              'layer2_block0', 'layer2_block1', 'layer2_block2', 'layer2_block3',
              'layer2', 
              'layer3_block0', 'layer3_block1', 'layer3_block2', 'layer3_block3',
              'layer3_block4', 'layer3_block5', 'layer3_block6', 'layer3_block7',
              'layer3_block8', 'layer3_block9', 'layer3_block10', 'layer3_block11',
              'layer3_block12', 'layer3_block13', 'layer3_block14', 'layer3_block15',
              'layer3_block16', 'layer3_block17', 'layer3_block18', 'layer3_block19',
              'layer3_block20', 'layer3_block21', 'layer3_block22', 
              'layer3', 
              'layer4_block0', 'layer4_block1', 'layer4_block2',
              'layer4']
    ids = [str(i) for i in range(10)]

    def compute_agg(dct, field):
        values = []
        for ly in layers:
            layer_vals = []
            for i in ids:
                v = dct[i][ly][field]
                if np.isnan(v):
                    v = 0.0
                layer_vals.append(v)
            values.append(np.mean(layer_vals))
        return values

    intra_mean = compute_agg(intra, 'mean')
    inter_mean = compute_agg(inter, 'mean')
    intra_std  = compute_agg(intra, 'std')
    inter_std  = compute_agg(inter, 'std')
    gap_mean   = (np.array(inter_mean) - np.array(intra_mean)).tolist()

    plot_dict = {
        "intra_mean": intra_mean,
        "inter_mean": inter_mean,
        "intra_std": intra_std,
        "inter_std": inter_std,
        "gap_mean": gap_mean,
        "gap_std": None
    }
    return plot_dict

In [13]:
plot_dicts = [get_plot_dict(epoch_data[e]['distance']) for e in plot_epochs_found]


In [ ]:
def plot_split(ax, xs, ys, color, marker, label, lw=2, alpha_tail=0.15, split_idx=5):
    ax.plot(xs, ys, color=color, linewidth=lw, label=label,marker=marker)


In [ ]:
def plot_state_lines_intra_inter(plot_dicts, plot_epochs, plot_f1s=None,
                                save_path=None, title=""):

    layers = ['input', 
              'layer1_block0', 'layer1_block1', 'layer1_block2',
              'layer1', 
              'layer2_block0', 'layer2_block1', 'layer2_block2', 'layer2_block3',
              'layer2', 
              'layer3_block0', 'layer3_block1', 'layer3_block2', 'layer3_block3',
              'layer3_block4', 'layer3_block5', 'layer3_block6', 'layer3_block7',
              'layer3_block8', 'layer3_block9', 'layer3_block10', 'layer3_block11',
              'layer3_block12', 'layer3_block13', 'layer3_block14', 'layer3_block15',
              'layer3_block16', 'layer3_block17', 'layer3_block18', 'layer3_block19',
              'layer3_block20', 'layer3_block21', 'layer3_block22', 
              'layer3', 
              'layer4_block0', 'layer4_block1', 'layer4_block2',
              'layer4']
    xs = list(range(len(layers)))
    
    fig, ax = plt.subplots(figsize=(5, 4))

    if plot_f1s is None:
        plot_f1s = [None] * len(plot_dicts)

    colors = plt.cm.Blues(np.linspace(0.30, 0.9, len(plot_dicts)))
    for plot_dict, epoch, f1, color in zip(plot_dicts, plot_epochs, plot_f1s, colors):
        mean = np.array(plot_dict["intra_mean"])
        if f1 is None:
            label = f"Epoch {epoch}"
        else:
            label = f"Epoch {epoch} (F1={f1:.4f})"
        plot_split(ax, xs, mean, color=color, marker=None, label=label)

    stage_labels = ["stage1", "stage2", "stage3", "stage4"]
    stage_positions = {label: [] for label in stage_labels}
    for i, layer in enumerate(layers):
        if layer == "input":
            stage_positions["stage1"].append(i)
        elif layer.startswith("layer1"):
            stage_positions["stage1"].append(i)
        elif layer.startswith("layer2"):
            stage_positions["stage2"].append(i)
        elif layer.startswith("layer3"):
            stage_positions["stage3"].append(i)
        elif layer.startswith("layer4"):
            stage_positions["stage4"].append(i)

    stage_ranges = []
    for label in stage_labels:
        positions = stage_positions[label]
        if positions:
            stage_ranges.append((label, min(positions), max(positions)))

    stage_colors = ["#EDFBE3", "#D8E6F9", "#F7DCD1", "#E3D1FA"]
    for (label, start, end), color in zip(stage_ranges, stage_colors):
        ax.axvspan(start - 0.5, end + 0.5, color=color, alpha=0.2, zorder=0, linewidth=0)

    stage_ticks = [(start + end) / 2 for _, start, end in stage_ranges]
    ax.set_xticks(stage_ticks)
    ax.set_xticklabels([label for label, _, _ in stage_ranges])
    ax.set_axisbelow(True)
    ax.grid(False)
    ax.legend()
    ax.set_title(title)
    plt.tight_layout()
    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close()


In [16]:
plot_state_lines_intra_inter(
    plot_dicts, plot_epochs_found, plot_f1s,
    save_path=f"../../results/figure_results/2_in_one.pdf",
    title=f"Intra-class mean distance"
    )
